# 第 14 章:智能体 RL —— 多轮工具使用作为轨迹

前几章的 RL(PPO/GRPO/CISPO)都是**单轮**的:模型生成一个回复,获得一个奖励。但真实的智能体需要**多轮交互**:

1. 用户提问:「北京明天天气如何?」
2. 模型生成:`<tool_call>{"name":"weather","arguments":{"city":"北京"}}</tool_call>`
3. 工具返回:`<tool_response>{"temp":25,"condition":"晴"}</tool_response>`
4. 模型基于结果继续生成:`北京明天晴,气温约 25°C。`

整个多轮交互构成一条**轨迹(trajectory)**,奖励在轨迹结束后才计算。这就是 **Agentic RL**。

## 14.1 工具调用格式

minimind 使用结构化的工具调用格式:

```
用户: 北京明天天气如何?

模型生成:
<tool_call>{"name": "weather", "arguments": {"city": "北京"}}</tool_call>

工具执行后注入:
<tool_response>{"temp": 25, "condition": "晴"}</tool_response>

模型继续生成:
根据查询,北京明天天气晴朗,气温约 25°C,适合外出活动。
```

`<tool_call>` 和 `<tool_response>` 都是 tokenizer 的**特殊 token**(id=21/22 和 23/24),不会被 BPE 拆分。

## 14.2 Mock 工具:训练时的「沙盒」

训练时不能调用真实 API(不稳定、有延迟、有费用)。minimind 实现了 6 个**确定性 mock 工具**(`train_agent.py:~40-64 (@67f114a)`):

| 工具 | 功能 | Mock 行为 |
|---|---|---|
| `math` | 数学计算 | 确定性公式 |
| `unit_convert` | 单位换算 | 固定换算表 |
| `weather` | 天气查询 | 固定数据 |
| `time` | 时间查询 | 固定时刻 |
| `fx` | 汇率 | 固定汇率 |
| `translate` | 翻译 | 固定译文 |

> **为什么 mock?** RL 训练需要成千上万次 rollout。真实 API 不可复现(同一输入可能返回不同结果),mock 保证确定性 —— 同样的 tool_call 永远返回同样的 response。

## 14.3 多轮 Rollout

`rollout_single`(`train_agent.py:~98-157 (@67f114a)`)是 Agentic RL 的核心。它执行一个完整的多轮交互:

```
Round 1:
  prompt → model.generate → 检查输出是否含 <tool_call>
    ├── 有 → 解析 tool_call → 执行 mock tool → 拼接 <tool_response>
    │        → 继续 Round 2
    └── 无 → 直接结束(不需要工具)

Round 2:
  (上轮的 <tool_response> + 生成提示) → model.generate
    ├── 有 → 解析 → 执行 → Round 3
    └── 无 → 最终回复

最多 max_turns=3 轮
```

整条轨迹的 `response_ids` 和 `response_mask` 被合并:

```python
# 所有轮次的 assistant 生成部分 → response_mask = 1(参与 RL loss)
# 工具返回部分 → response_mask = 0(不参与,因为是外部注入的)
response_ids = torch.cat([round1_ids, round2_ids, round3_ids])
response_mask = torch.cat([round1_mask, round2_mask, round3_mask])
```

## 14.4 延迟奖励(Delayed Reward)

与单轮 RL 不同,Agentic RL 的奖励在**整条轨迹结束后**才计算:

```
单轮 RL:  prompt → response → reward(立即)
多轮 RL:  prompt → tool_call → tool_response → response → tool_call → ... → reward(延迟)
```

**为什么延迟?** 因为中间步骤的质量取决于后续结果。比如:

- Round 1 调用了错误的工具 → Round 2 得到无用的 response → 最终答案错误
- 只有看到最终结果,才能判断 Round 1 的工具调用是否合理

> 这就是 **credit assignment** 难题:最终奖励应该怎么分配到每个 token?GRPO/CISPO 的答案是:整条轨迹共享同一个优势(组归一化后的 reward)。

## 14.5 奖励函数

`calculate_rewards`(`train_agent.py:~188-239 (@67f114a)`)组合多个维度:

| 维度 | 分数 | 说明 |
|---|---|---|
| **工具调用有效性** | 0~1 | `<tool_call>` 格式是否正确(JSON 可解析) |
| **gt 命中率** | `2.5 × verified/gt` | 调用的工具返回了正确的 ground-truth 结果 |
| **格式闭合** | ±0.5 | `<tool_call>` 与 `</tool_call>` 是否配对 |
| **未完成惩罚** | -1 | 超长截断(没生成完) |
| **RM 分数** | [-3, 3] | 奖励模型的连续评分 |

最终 reward clamp 到 [-3, 3]。

```python
reward = (
    tool_validity      # 0~1
    + 2.5 * len(verified) / len(gt)  # gt 命中率
    + format_closure   # ±0.5
    - unfinished_penalty  # -1 if truncated
    + rm_score         # [-3, 3]
)
reward = max(-3.0, min(3.0, reward))
```

## 14.6 AgentRLDataset

读 `lm_dataset.py:226-252`:

```python
# 数据格式
{
    "conversations": [...],   # 对话历史
    "gt": ["25°C", "晴"]     # ground-truth 答案列表(用于校验工具结果)
}
```

`gt` 是预先标注的正确答案列表。当工具返回的结果命中了 `gt` 中的条目,reward 增加。这提供了一种**弱监督**:不需要精确标注每一步,只需要标注最终期望的结果。

## 14.7 训练配置

Agentic RL 使用与 GRPO 相同的 loss(cispo/grpo),但应用于多轮轨迹:

```python
# CISPO/GRPO loss — train_agent.py:~321-328 (@67f114a) (与 train_grpo.py 相同的 loss)
ratio = torch.exp(new_logps - old_logps)
if loss_type == 'cispo':
    clipped = torch.clamp(ratio, max=epsilon_high)
    loss = -(clipped * advantages * new_logps).mean()
else:  # grpo
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1-eps, 1+eps) * advantages
    loss = -torch.min(surr1, surr2).mean()
loss += beta * kl.mean()
```

关键区别:**response_mask 只覆盖 assistant 生成的 token**,不覆盖工具返回的 `<tool_response>`(因为模型不应该为外部注入的内容负责)。

> **🔧 后缀逻辑更新**: Agent-RL checkpoint 保存路径(`ckp` — train_agent.py:~353 (@67f114a))同样走 `_model_suffix(lm_config)` (见 [ch08 §8.8](../ch08/01_main-chapter-code/ch08.ipynb)), 按 `_ple`/`_moe`/空 区分。**默认 Dense 模式后缀为空**, checkpoint 路径与上游原版一致。

&nbsp;

---

## Summary and takeaways

- Agentic RL = 多轮工具调用作为一条 RL 轨迹
- **Mock 工具**保证训练的确定性和可复现性
- **延迟奖励**:整条轨迹结束后才算 reward
- **gt 命中率**:用弱监督(只标注期望结果)提供训练信号
- response_mask 只覆盖 assistant 生成,不覆盖 tool_response

> **核心认知**:智能体不是「调 API 的胶水代码」,而是一条从感知到行动的优化轨迹。RL 让模型学会**何时调用工具、调用什么工具、如何利用结果**。

- 精简复习版见 [`./agent-rl.ipynb`](./agent-rl.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)

下一章:[第 15 章 · 知识蒸馏与 MoE](../ch15/01_main-chapter-code/README.md)